# Full-Frame Extraction For AOI Pipeline

Run this notebook on the machine that holds the Tobii scene videos and the per-participant CSV exports.

Instead of extracting only fixation frames, it extracts every video frame or sampled video frames and builds a frame-to-Tobii alignment table.

Outputs are written in a separate folder so the old fixation-frame pipeline stays untouched.

In [ ]:
from pathlib import Path
import re
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

print('cv2', cv2.__version__)
print('pandas', pd.__version__)

## Configuration

Edit these paths before running.

`TARGET_FPS = None` extracts every native video frame. For a smaller upload, set `TARGET_FPS = 5`, `10`, etc.

In [ ]:
# Folder containing I1_Day, I1_Night, I3_Day, ... participant CSVs
SEPARATE_EXCELS_ROOT = Path("/path/to/Separate_excels")

# Folder containing Tobii session folders named like <date>_<time>_<PARTICIPANT_ID>.day / .night, each with scenevideo.mp4
VIDEO_ROOT = Path("/path/to/scene_videos")

# New output folder for full-frame extraction
OUTPUT_ROOT = Path("/path/to/AOI_full_frame_output")

# None means every frame. Otherwise sample at this FPS.
TARGET_FPS = None  # examples: None, 5, 10

# For separated CSVs, extract only the video interval covered by the CSV timestamps.
# Example: 364001945 microseconds starts extraction near 364.002 seconds, not video time 0.
EXTRACT_ONLY_CSV_TIME_RANGE = True
TIME_RANGE_PADDING_SEC = 0.25

# jpg is much smaller than png and is enough for GroundingDINO.
IMAGE_EXT = ".jpg"
JPEG_QUALITY = 92

# Optional subset controls while testing. Use [] for all.
CONDITIONS_TO_PROCESS = []  # example: ["I1_Day"]
PARTICIPANTS_TO_PROCESS = []  # example: ["PARTICIPANT_ID"]
MAX_PARTICIPANTS_PER_CONDITION = None

# Tobii timestamp column, in microseconds since video start.
TOBII_TIMESTAMP_COL = "Recording timestamp"

# Keep unmatched frames in alignment CSV, marked False.
KEEP_UNMATCHED_FRAMES = False

# Rename images after alignment so the first ts_<number> in the filename is the Tobii CSV timestamp.
# This keeps compatibility with the old GroundingDINO parsing/merge logic.
RENAME_IMAGES_TO_TOBII_TIMESTAMP = True
DROP_UNMATCHED_FRAME_IMAGES = True

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Output root:', OUTPUT_ROOT)

## Helper Functions

These mirror the old notebook's session matching logic, but produce a frame manifest and a nearest-time Tobii alignment table.

In [ ]:
def infer_time_of_day_from_condition(condition_name: str) -> str:
    name = condition_name.lower()
    if "night" in name:
        return "night"
    if "day" in name:
        return "day"
    return "unknown"


def participant_id_from_csv(csv_path: Path) -> str:
    return csv_path.stem.split('.')[0]


def suffix_from_csv(csv_path: Path, time_of_day: str) -> str:
    parts = csv_path.stem.split('.')
    if len(parts) > 1 and parts[-1] in {"d", "n"}:
        return parts[-1]
    return "n" if time_of_day == "night" else "d"


def find_all_video_dirs(participant_id: str, time_of_day: str):
    suffix = f"_{participant_id}.{time_of_day}"
    return sorted(
        [d for d in VIDEO_ROOT.iterdir() if d.is_dir() and d.name.endswith(suffix)],
        key=lambda p: p.name,
    )


def video_metadata(video_path: Path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    duration = frame_count / fps if fps > 0 else np.nan
    return {
        "fps": fps,
        "frame_count": frame_count,
        "width": width,
        "height": height,
        "duration_sec": duration,
    }


def load_tobii_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    if TOBII_TIMESTAMP_COL not in df.columns:
        raise ValueError(f"Missing {TOBII_TIMESTAMP_COL!r} in {csv_path}")
    df = df.copy()
    df[TOBII_TIMESTAMP_COL] = pd.to_numeric(df[TOBII_TIMESTAMP_COL], errors="coerce")
    df = df.dropna(subset=[TOBII_TIMESTAMP_COL]).sort_values(TOBII_TIMESTAMP_COL)
    df[TOBII_TIMESTAMP_COL] = df[TOBII_TIMESTAMP_COL].astype(np.int64)
    return df


def choose_best_session(matches, tobii_df: pd.DataFrame) -> Path:
    if len(matches) == 1:
        return matches[0]

    expected = int(tobii_df[TOBII_TIMESTAMP_COL].max()) / 1e6
    scored = []
    for d in matches:
        vp = d / "scenevideo.mp4"
        if not vp.exists():
            continue
        meta = video_metadata(vp)
        if not meta or not np.isfinite(meta["duration_sec"]):
            continue
        scored.append((abs(meta["duration_sec"] - expected), meta["duration_sec"], d))

    if not scored:
        raise RuntimeError(f"No readable scenevideo.mp4 among candidates: {[m.name for m in matches]}")

    scored.sort(key=lambda x: x[0])
    if len(scored) > 1 and abs(scored[1][0] - scored[0][0]) < 1.0:
        raise RuntimeError(
            "Ambiguous session choice. "
            f"Expected duration ~ {expected:.2f}s. "
            f"Candidates: {[(s[2].name, round(s[1], 2), round(s[0], 2)) for s in scored[:3]]}"
        )
    return scored[0][2]


def selected_csvs():
    condition_dirs = sorted([d for d in SEPARATE_EXCELS_ROOT.iterdir() if d.is_dir()], key=lambda p: p.name)
    if CONDITIONS_TO_PROCESS:
        wanted = set(CONDITIONS_TO_PROCESS)
        condition_dirs = [d for d in condition_dirs if d.name in wanted]

    for condition_dir in condition_dirs:
        csvs = sorted(condition_dir.glob("*.csv"), key=lambda p: p.name)
        if PARTICIPANTS_TO_PROCESS:
            wanted_pids = set(str(x) for x in PARTICIPANTS_TO_PROCESS)
            csvs = [p for p in csvs if participant_id_from_csv(p) in wanted_pids]
        if MAX_PARTICIPANTS_PER_CONDITION is not None:
            csvs = csvs[:MAX_PARTICIPANTS_PER_CONDITION]
        for csv_path in csvs:
            yield condition_dir.name, csv_path

## Frame Extraction And Alignment Functions

The extractor reads sequentially rather than seeking for every frame, which is much better for full-video extraction.

In [ ]:
def should_save_frame(native_idx: int, native_fps: float, target_fps):
    if target_fps is None:
        return True
    if target_fps <= 0:
        raise ValueError("TARGET_FPS must be positive or None")
    step = max(1, round(native_fps / float(target_fps)))
    return native_idx % step == 0


def write_frame(path: Path, frame) -> bool:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() in {".jpg", ".jpeg"}:
        return bool(cv2.imwrite(str(path), frame, [int(cv2.IMWRITE_JPEG_QUALITY), int(JPEG_QUALITY)]))
    return bool(cv2.imwrite(str(path), frame))


def extract_video_frames(
    video_path: Path,
    frames_dir: Path,
    participant_id: str,
    time_of_day: str,
    start_time_sec=None,
    end_time_sec=None,
):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if fps <= 0:
        raise RuntimeError(f"Video has invalid FPS {fps}: {video_path}")

    start_native_idx = 0 if start_time_sec is None else max(0, int(np.floor(float(start_time_sec) * fps)))
    if total_frames > 0:
        default_end_idx = total_frames - 1
        end_native_idx = default_end_idx if end_time_sec is None else min(default_end_idx, int(np.ceil(float(end_time_sec) * fps)))
    else:
        end_native_idx = None if end_time_sec is None else int(np.ceil(float(end_time_sec) * fps))

    if end_native_idx is not None and start_native_idx > end_native_idx:
        cap.release()
        raise ValueError(f"Empty extraction window: {start_time_sec=} {end_time_sec=}")

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_native_idx)
    rows = []
    saved = 0
    read_count = 0
    native_idx = start_native_idx
    frames_to_read = None if end_native_idx is None else max(0, end_native_idx - start_native_idx + 1)

    pbar = tqdm(total=frames_to_read, desc=f"extract {participant_id}.{time_of_day}")
    while True:
        if end_native_idx is not None and native_idx > end_native_idx:
            break

        ret, frame = cap.read()
        if not ret or frame is None:
            break
        read_count += 1

        if should_save_frame(native_idx, fps, TARGET_FPS):
            frame_time_sec = native_idx / fps if fps > 0 else np.nan
            frame_timestamp_us = int(round(frame_time_sec * 1_000_000)) if np.isfinite(frame_time_sec) else -1
            image_name = f"{participant_id}.{time_of_day}_frame_{native_idx:07d}_ts_{frame_timestamp_us}{IMAGE_EXT}"
            image_path = frames_dir / image_name
            if write_frame(image_path, frame):
                saved += 1
                rows.append({
                    "image_path": str(image_path),
                    "image_name": image_name,
                    "participant_id": participant_id,
                    "time_of_day": time_of_day,
                    "native_frame_index": native_idx,
                    "frame_time_sec": frame_time_sec,
                    "frame_timestamp_us": frame_timestamp_us,
                    "source_video_path": str(video_path),
                    "source_fps": fps,
                    "source_width": width,
                    "source_height": height,
                    "target_fps": TARGET_FPS if TARGET_FPS is not None else fps,
                    "extraction_start_sec": start_time_sec,
                    "extraction_end_sec": end_time_sec,
                })

        native_idx += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    return pd.DataFrame(rows), saved, read_count


def align_manifest_to_tobii(frame_manifest: pd.DataFrame, tobii_df: pd.DataFrame, tolerance_sec=None):
    frames = frame_manifest.sort_values("frame_timestamp_us").copy()
    tobii = tobii_df.sort_values(TOBII_TIMESTAMP_COL).copy()

    aligned = pd.merge_asof(
        frames,
        tobii,
        left_on="frame_timestamp_us",
        right_on=TOBII_TIMESTAMP_COL,
        direction="nearest",
        suffixes=("", "_tobii"),
    )

    aligned["tobii_time_delta_sec"] = (
        aligned["frame_timestamp_us"] - aligned[TOBII_TIMESTAMP_COL]
    ).abs() / 1_000_000.0

    if tolerance_sec is None:
        fps = float(frames["target_fps"].dropna().iloc[0]) if len(frames) else 30.0
        tolerance_sec = 0.5 / fps if fps > 0 else None

    if tolerance_sec is not None:
        aligned["tobii_match_within_tolerance"] = aligned["tobii_time_delta_sec"] <= tolerance_sec
    else:
        aligned["tobii_match_within_tolerance"] = True

    if not KEEP_UNMATCHED_FRAMES and not DROP_UNMATCHED_FRAME_IMAGES:
        aligned = aligned[aligned["tobii_match_within_tolerance"]].copy()

    aligned["tobii_alignment_tolerance_sec"] = tolerance_sec
    return aligned


def apply_tobii_timestamp_filenames(aligned: pd.DataFrame) -> pd.DataFrame:
    if not RENAME_IMAGES_TO_TOBII_TIMESTAMP:
        return aligned

    rows = []
    for _, row in aligned.iterrows():
        source_path = Path(row["image_path"])
        matched = bool(row.get("tobii_match_within_tolerance", False))
        if not matched:
            if DROP_UNMATCHED_FRAME_IMAGES and source_path.exists():
                source_path.unlink()
            if not DROP_UNMATCHED_FRAME_IMAGES:
                rows.append(row)
            continue

        tobii_ts = int(row[TOBII_TIMESTAMP_COL])
        native_ts = int(row["frame_timestamp_us"])
        native_idx = int(row["native_frame_index"])
        participant_id = row["participant_id"]
        time_of_day = row["time_of_day"]
        new_name = f"{participant_id}.{time_of_day}_ts_{tobii_ts}_frame_ts_{native_ts}_frame_{native_idx:07d}{source_path.suffix}"
        target_path = source_path.with_name(new_name)

        dup_idx = 1
        while target_path.exists() and target_path.resolve() != source_path.resolve():
            new_name = f"{participant_id}.{time_of_day}_ts_{tobii_ts}_frame_ts_{native_ts}_frame_{native_idx:07d}_dup{dup_idx}{source_path.suffix}"
            target_path = source_path.with_name(new_name)
            dup_idx += 1

        if source_path.exists() and source_path.resolve() != target_path.resolve():
            source_path.rename(target_path)

        row = row.copy()
        row["native_frame_timestamp_us"] = native_ts
        row["matched_tobii_timestamp_us"] = tobii_ts
        row["grounding_merge_timestamp_us"] = tobii_ts
        row["native_image_name"] = row["image_name"]
        row["image_name"] = new_name
        row["image_path"] = str(target_path)
        rows.append(row)

    return pd.DataFrame(rows)


## Run A Small Test First

Use this cell to test one participant before running the batch. Change the values if needed.

In [ ]:
TEST_CONDITION = "I1_Day"
TEST_CSV_NAME = "PARTICIPANT_ID.d.csv"

def process_one(condition_name: str, csv_path: Path):
    time_of_day = infer_time_of_day_from_condition(condition_name)
    if time_of_day == "unknown":
        raise ValueError(f"Cannot infer day/night from {condition_name}")

    participant_id = participant_id_from_csv(csv_path)
    suffix = suffix_from_csv(csv_path, time_of_day)
    tobii_df = load_tobii_csv(csv_path)
    csv_start_sec = float(tobii_df[TOBII_TIMESTAMP_COL].min()) / 1_000_000.0
    csv_end_sec = float(tobii_df[TOBII_TIMESTAMP_COL].max()) / 1_000_000.0
    extraction_start_sec = max(0.0, csv_start_sec - TIME_RANGE_PADDING_SEC) if EXTRACT_ONLY_CSV_TIME_RANGE else None
    extraction_end_sec = (csv_end_sec + TIME_RANGE_PADDING_SEC) if EXTRACT_ONLY_CSV_TIME_RANGE else None

    matches = find_all_video_dirs(participant_id, time_of_day)
    if not matches:
        raise RuntimeError(f"No video session found for {participant_id}.{time_of_day}")
    session_dir = choose_best_session(matches, tobii_df)
    video_path = session_dir / "scenevideo.mp4"

    out_participant_dir = OUTPUT_ROOT / condition_name / f"{participant_id}.{suffix}"
    frames_dir = out_participant_dir / "frames"
    out_participant_dir.mkdir(parents=True, exist_ok=True)

    print(f"Processing {condition_name}/{participant_id}.{suffix}")
    print(f"CSV: {csv_path}")
    print(f"Video: {video_path}")
    print(f"CSV time window: {csv_start_sec:.6f}s -> {csv_end_sec:.6f}s")
    if EXTRACT_ONLY_CSV_TIME_RANGE:
        print(f"Extraction window: {extraction_start_sec:.6f}s -> {extraction_end_sec:.6f}s")
    print(f"Output: {out_participant_dir}")

    manifest, saved, read_count = extract_video_frames(
        video_path,
        frames_dir,
        participant_id,
        time_of_day,
        start_time_sec=extraction_start_sec,
        end_time_sec=extraction_end_sec,
    )
    aligned = align_manifest_to_tobii(manifest, tobii_df)
    aligned = apply_tobii_timestamp_filenames(aligned)
    manifest = aligned.copy()

    manifest_path = out_participant_dir / "frame_manifest.csv"
    manifest.to_csv(manifest_path, index=False)

    aligned_path = out_participant_dir / "frame_tobii_alignment.csv"
    aligned.to_csv(aligned_path, index=False)

    return {
        "condition": condition_name,
        "participant_id": participant_id,
        "suffix": suffix,
        "video_path": str(video_path),
        "csv_start_sec": csv_start_sec,
        "csv_end_sec": csv_end_sec,
        "extraction_start_sec": extraction_start_sec,
        "extraction_end_sec": extraction_end_sec,
        "frames_read": read_count,
        "raw_frames_saved_before_alignment_filter": saved,
        "frames_saved": int(len(aligned)),
        "manifest_path": str(manifest_path),
        "alignment_path": str(aligned_path),
        "matched_frames": int(aligned["tobii_match_within_tolerance"].sum()),
        "total_aligned_rows": int(len(aligned)),
    }


test_csv = SEPARATE_EXCELS_ROOT / TEST_CONDITION / TEST_CSV_NAME
if test_csv.exists():
    test_summary = process_one(TEST_CONDITION, test_csv)
    print(test_summary)
else:
    print('Test CSV not found:', test_csv)

## Batch Run

Run this after the small test looks right. The output folder can be zipped and copied to the GPU machine.

In [ ]:
RUN_BATCH = False  # change to True when ready

if RUN_BATCH:
    summary_rows = []
    error_rows = []

    for condition_name, csv_path in selected_csvs():
        try:
            summary_rows.append(process_one(condition_name, csv_path))
        except Exception as e:
            error_rows.append({
                "condition": condition_name,
                "csv_path": str(csv_path),
                "participant_id": participant_id_from_csv(csv_path),
                "error": str(e),
            })
            print('ERROR:', condition_name, csv_path.name, e)

    summary_df = pd.DataFrame(summary_rows)
    errors_df = pd.DataFrame(error_rows)
    summary_path = OUTPUT_ROOT / "batch_summary.csv"
    errors_path = OUTPUT_ROOT / "batch_errors.csv"
    summary_df.to_csv(summary_path, index=False)
    errors_df.to_csv(errors_path, index=False)

    print('Done')
    print('Summary:', summary_path)
    print('Errors:', errors_path)
else:
    print('RUN_BATCH is False. Set it to True when ready.')

## Copy To The GPU Machine

Copy the output folder to the machine that runs the detectors. It becomes the `AOI_full_frame_output` folder under `AOI_DATA_ROOT`:

```text
$AOI_DATA_ROOT/AOI_full_frame_output/<condition>/<participant>.<d|n>/
```

The important files per participant are:

```text
frames/*.jpg
frame_manifest.csv
frame_tobii_alignment.csv
```

The next step is `run_full_frame_groundingdino.py` (or `run_full_frame_finetuned.py`) on `frames/`, which includes the same-category box merge.